In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import gc 
gc.collect()

47

In [3]:
import pandas as pd
import numpy as np
import getpass
import sys
import datetime
import io
usr_name = getpass.getuser()
sys.path.append(f'/home/{usr_name}/notebooks/utils')
from spark_utils import *
#import datadicts as dd
from bpm_features import calc_all_features

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [4]:
import os
import sys
import numpy as np
import pandas as pd
# import polars as pl
  
def get_spark_session(name, level):
    """
    Get spark context
    :: name - set your app name
    :: level - set max resources level
    """
    python_path = sys.executable
    kernel = python_path.split('/')[-3]
    os.environ['SPARK_MAJOR_VERSION'] = '3'
    os.environ['SPARK_HOME'] = '/usr/sdp/current/spark3-client/'
    os.environ['PYSPARK_DRIVER_PYTHON'] = python_path
    os.environ['PYSPARK_PYTHON'] = python_path
    os.environ['LD_LIBRARY_PATH'] = '/opt/python/virtualenv/jupyter/lib'
    sys.path.insert(0, '/usr/sdp/current/spark3-client/python/')
    sys.path.insert(0, '/usr/sdp/current/spark3-client/python/lib/py4j_current')
 
    # Resources Level Profiles                           #  cpu --  ram -- desc
    if level == 1: lv = ['basic',2,10,2,10,2,2,10]       #   21 --  142 -- для базовых запросов (show create table tbl, show partitions tbl)
    if level == 2: lv = ['basic+CPU',2,10,2,10,2,2,20]   #   41 --  262 -- для простой аналитики (select * from limit 100, sum/count/avg)
    if level == 3: lv = ['middle',4,28,6,28,6,4,20]      #   81 --  742 -- для агрегатов за период 1-2мес (client_aggr_mnth, epk_campaign_daily)
    if level == 4: lv = ['middle+CPU',4,28,6,28,6,4,25]  #  101 --  912 -- для агрегатов за период >1-6мес  (client_aggr_mnth, epk_campaign_daily)
    if level == 5: lv = ['high',4,28,6,36,8,6,30]        #  121 -- 1100 -- для детальных таблиц с большими партициями (_sbol, _card, _eps)
    if level == 6: lv = ['high+CPU',4,18,5,36,8,6,40]    #  161 -- 1000 -- для детальных таблиц с мелкими партициями (feedbacks)
    if level == 7: lv = ['unfriendly',5,28,6,44,10,8,40] #  201 -- 1458 -- для запуска вечером/ночью или на пустом кластере (не рекомендуется)
    lvname = f'{level}.{lv[0]}({lv[1]*lv[7]+1},{lv[4]+lv[2]*lv[7]})'
    print(f'Kernel: {kernel}, Python_path: {python_path}, Resource_level: {lvname}')
    
    # Spark Config      
    from pyspark import SparkContext, SparkConf
    from pyspark.sql import SparkSession
  
    conf = SparkConf().setAppName(f'{name} \n ::{kernel}::{lvname}::')\
        .setMaster("yarn")\
        .set('spark.executor.cores',                     f'{lv[1]}')\
        .set('spark.executor.memory',                    f'{lv[2]}g')\
        .set('spark.executor.memoryOverhead',            f'{lv[3]}g')\
        .set('spark.driver.memory',                      f'{lv[4]}g')\
        .set('spark.driver.memoryOverhead',              f'{lv[5]}g')\
        .set('spark.driver.maxResultSize', '10g')\
        .set('spark.dynamicAllocation.initialExecutors', f'{lv[6]}')\
        .set('spark.dynamicAllocation.maxExecutors',     f'{lv[7]}')\
        .set('spark.dynamicAllocation.enabled', 'true')\
        .set('spark.dynamicAllocation.executorIdleTimeout', '120s')\
        .set('spark.dynamicAllocation.cachedExecutorIdleTimeout', '600s')\
        .set('spark.hive.mapred.supports.subdirectories', 'true')\
        .set('spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive', 'true')\
        .set('spark.shuffle.service.enabled', 'true')\
        .set('spark.port.maxRetries', '150')\
       .set('spark.sql.parquet.writeLegacyFormat', 'true')\
        .set('spark.kerberos.access.hadoopFileSystems','hdfs://arnsdpsbx:8020/')\
        .set('spark.sql.autoBroadcastJoinThreshold','20971520')
    
    spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()
    return spark

try: spark
except NameError: print('Spark3 Starting')
else:
    print('Spark3 Restarting')
    spark.stop()
    
spark = get_spark_session('platon_features_premier', 6) # For example, MyPySpark3
  
import pyspark.sql.functions as sf
from pyspark.sql.types import *
  
sc = spark.sparkContext
sc.setLogLevel('OFF')  # or 'INFO' or 'WARN' or 'OFF'
spark


Spark3 Starting
Kernel: mlpy3811v23, Python_path: /data/sdp/mlpy3811v23/bin/python, Resource_level: 6.high+CPU(161,756)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/16 18:39:48 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/10/16 18:39:48 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/10/16 18:39:48 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/10/16 18:39:48 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/10/16 18:39:48 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
25/10/16 18:39:48 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
25/10/16 18:40:14 WARN HiveConf: HiveConf of name hive.mapred.supports.subdirectories does not exist
25/10/16 18:40:24 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


# Train

In [ ]:
target = spark.sql('''
select epk_id, last_day(date(report_dt) - interval 3 month) as report_dt, target
from arnsdpsbx_team_ss.bpm_premier_target_2025_04_30_multiclass
''')

target.createOrReplaceTempView('target')
target.count()
target.show()

In [6]:
tables_dict = {'agg':'prx_bpm_client_aggr_custom_rozn_client_aggr.ft_client_aggr_mnth',
               'feedbacks':'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_feedbacks',
               'vsp_visits':'prx_bpm_visiting_vsp_custom_rozn_sscxdata_cxdm.cxdm_visiting_vsp_v2',
               'card_transactions': 'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_card_transactions',
               'e_cod':'prx_bpm_cod_platform_cod.cod_deposit_deposit',
               'idoc': 'prx_bpm_arrests_internal_aiv_deposit.idoc',
               'idoc_acc': 'prx_bpm_arrests_internal_aiv_deposit.idoc_acc',
               'pos_embeddings_fl': 'prx_bpm_pos_emb_custom_rozn_ml360.u_fl_transaction_embeddings' ,
               'embeddings_fl': 'prx_bpm_multimodel_emb_custom_fin_palm_ml.cmn_multimodal_emb_ind_fct'}


In [7]:
model_name = 'bpm_premier_multiclass'
output_scheme = 'arnsdpsbx_team_ss' 
mode = 'append'

In [8]:
calc_all_features(spark, target, tables_dict, model_name, output_scheme, mode)

agg_features: done
arrests_features: done
feedbacks_features: done
card_transactions_features: done
flows12_features: done


pos_dynamic_features: done


pos_embeddings_fl_features: done


all_features_train: done
arnsdpsbx_team_ss.bpm_premier_multiclass_all_features_train


100%|██████████| 133/133 [05:17<00:00,  2.39s/it]


all_features_fix: done
arnsdpsbx_team_ss.bpm_premier_multiclass_all_features_fix_types_train: done


In [6]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
import pyspark.sql.types as T

def fix_spark_types(df):
    for col, dtype in df.dtypes:
            if "decimal" in dtype:
                df = df.withColumn(col, F.col(col).cast(T.DoubleType()))

    date_columns = [i for i in df.columns if '_dt' in i]
    for col in date_columns:
        df = df.withColumn(col, F.when(F.col(col) > F.to_date(F.lit(pd.Timestamp.max)), F.to_date(F.lit(pd.Timestamp.max))).otherwise(F.col(col)))
    
    return df

In [7]:
last = spark.sql('''
select * from arnsdpsbx_team_ss.bpm_premier_multiclass_all_features_train
''')


all_features_fix = fix_spark_types(last)
all_features_fix.write.saveAsTable('arnsdpsbx_team_ss.bpm_premier_multiclass_all_features_fix_types_train_new', mode='overwrite')

In [8]:
spark.sql('''
select * from arnsdpsbx_team_ss.bpm_premier_multiclass_all_features_fix_types_train_new
''').write.parquet('hdfs://arnsdpsbx/user/team/team_ss/bpm_premier_multiclass_all_features_fix_types_train_new_model', mode='overwrite')

# OOT

In [7]:
oot = spark.sql(f'''
with target as (select epk_id, last_day(date(report_dt) - interval 3 month) as report_dt, target
from arnsdpsbx_team_ss.bpm_premier_target_multiclass_oot
),

agg_1m as (select epk_id, report_dt, tp_active_kind_cd, sd_age_yrs_frac_nv, tp_1st_open_dt, insur_total_bal,
tp_best_kind_ever_cd, seg_client_fl_segment_cd, seg_age_segment, prl_client_app_income_amt,
dep_acct_dep_td_bal_rub_amt, dep_topup_12m_avg_rub_amt, lne_lst_open_pl_rate, dep_acct_dep_save_bal_rub_amt,
prd_lst_prod_tb_cd, dep_acct_tot_bal_prev_rub_amt, srv_ap_othr_1st_txn_ever_dt, lne_pl_clsd_wavg_intr_rate,
tp_mnth_lst_grace_end_exp_qty, dep_acct_dep_mnth_lst_open_qty, srv_sbol_mnth_web_lst_log_qty, 
tp_mnth_lst_close_qty, srv_sbol_mnth_1st_txn_qty, crd_dc_mnth_snc_open_qty, dep_acct_dep_td_qty
from prx_bpm_client_aggr_custom_rozn_client_aggr.ft_client_aggr_mnth
where report_dt = '2025-04-30'),

flows as (select epk_id, report_dt, (coalesce(dep_tot_bal_rub_amt, 0) +
                            coalesce(inv_mf_agrmnt_bal_rub_amt, 0) +
                            coalesce(inv_tm_agrmnt_bal_rub_amt, 0) +
                            coalesce(inv_bo_agrmnt_bal_tot_rub_amt, 0) +
                            coalesce(bal_invest_insur_life_amt, 0) +
                            coalesce(bal_nakop_insur_life_amt, 0)
                            ) as tot_bal_with_invest,
                        crd_otf_total_rub_amt
FROM
                prx_bpm_client_aggr_custom_rozn_client_aggr.ft_client_aggr_mnth
            WHERE
                report_dt BETWEEN '2024-05-31' and '2025-04-30'
                        
),

target_deep as (
            SELECT
                epk_id,
                report_dt,
                last_day(add_months(report_dt, -2)) AS report_dt_3m,
                last_day(add_months(report_dt, -11)) AS report_dt_12m
            FROM 
                target   
), 

flows_deep as (
            SELECT 
                target_deep.epk_id,
                target_deep.report_dt,
                avg(agg_12m.crd_otf_total_rub_amt) as crd_otf_total_rub_amt_12m, -- Σ списаний по всем картам
                avg(agg_12m.tot_bal_with_invest) as tot_bal_with_invest_12m,
                avg(agg_3m.tot_bal_with_invest) as tot_bal_with_invest_3m
            from 
                target_deep
                LEFT JOIN flows agg_3m USING (epk_id)
                LEFT JOIN flows agg_12m USING (epk_id)
            where 
                agg_3m.report_dt BETWEEN target_deep.report_dt_3m AND target_deep.report_dt
                AND agg_12m.report_dt BETWEEN target_deep.report_dt_12m AND target_deep.report_dt
            GROUP BY
                target_deep.report_dt,
                target_deep.epk_id
            ),
            
feedback_1m as (select epk_id, full_torg_pos_pos_12m, report_dt from prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_card_transactions
where report_dt = '2025-04-30'), 

feedback_12m as (select epk_id, sum(opened_feedbacks_cnt) as pfm_opened_cnt_all_12m, report_dt from prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_feedbacks
where report_dt between '2024-05-31' and '2025-04-30'
group by epk_id, report_dt)

select * from target
left join agg_1m using(epk_id, report_dt)
left join flows_deep using(epk_id, report_dt)
left join feedback_1m using(epk_id, report_dt)
left join feedback_12m using(epk_id, report_dt)
''')


all_features_fix = fix_spark_types(oot)
all_features_fix.write.saveAsTable('arnsdpsbx_team_ss.bpm_premier_multiclass_all_features_fix_types_oot', mode='overwrite')

In [ ]:
#собираем договора за 3 месяца, например between '2025-03' and '2025-05'

cod_dep_agrmnt_union = spark.read.table(f"prx_bpm_vitrina_na_dannyx_cod_custom_rb_cod.dep_agrmnt_13").filter(f"row_actual_from_month between concat(year('{date_now}'), '-05') and date_format('{date_now}', 'yyyy-MM') ")
print("13-й ЦОД договоров прочли")

all_paths = ['16', '18', '38', '40', '42', '44', '52', '54', '55', '70']

for i in all_paths:
    df_temp = spark.read.table(f"prx_bpm_vitrina_na_dannyx_cod_custom_rb_cod.dep_agrmnt_" + i).filter(f"row_actual_from_month between concat(year('{date_now}'), '-05') and date_format('{date_now}', 'yyyy-MM') ")
    cod_dep_agrmnt_union = cod_dep_agrmnt_union.unionAll(df_temp)
    print(f"{i}-й ЦОД договоров прочли")

cod_dep_agrmnt_union.createOrReplaceTempView("cod_dep_agrmnt_union")



tp_kind_name=spark.sql(f"""
select distinct agrmnt_product_name
from arnsdpsbx_team_ss.bpm_agent_py_levels_all_and_benefit a
left join cod_dep_agrmnt_union b on a.epk = b.epk_id
""")

tp_kind_name.createOrReplaceTempView("flags_deposit_savings")

In [8]:
spark.sql('''
select * from arnsdpsbx_team_ss.bpm_premier_multiclass_all_features_fix_types_oot
''').write.parquet('hdfs://arnsdpsbx/user/team/team_ss/bpm_premier_multiclass_all_features_fix_types_oot_model', mode='overwrite')